# task 1

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


alpha  = np.array([0.297104, 1.236745, 5.749982, 38.216677])

# Create Hamiltonian matrix
n = len(alpha)
h = np.zeros((n, n))
for p in range(n):
    for q in range(n):
        alpha_sum = alpha[p] + alpha[q]
        h[p][q] = 3 * np.pi * alpha[q] * np.sqrt(np.pi / alpha_sum**3) - \
                    3 * np.pi * alpha[q]**2 * np.sqrt(np.pi / alpha_sum**5) - \
                    4 * np.pi / alpha_sum

# Create overlap matrix
S = np.zeros((n, n))
for p in range(n):
    for q in range(n):
        S[p][q] = (np.pi / (alpha[p] + alpha[q]))**1.5

# Create Coulomb integral tensor
Q = np.zeros((n, n, n, n))
for p in range(n):
    for r in range(n):
        for q in range(n):
            for s in range(n):
                Q[p][r][q][s] = 2 * np.pi**2.5 / (
                    (alpha[p] + alpha[q]) * (alpha[r] + alpha[s]) *
                    np.sqrt(alpha[p] + alpha[q] + alpha[r] + alpha[s])
                )

# Pick initial value:
C = np.array([1, 1, 1, 1], dtype=np.float64)  # Ensure C is a float array

# Normalize coefficients using the overlap matrix
norm2 = 0
for p in range(n):
    for q in range(n):
        norm2 += C[p] * S[p][q] * C[q]
C /= np.sqrt(norm2)

# Compute initial energy
energy = 0
for p in range(n):
    for q in range(n):
        energy += 2 * C[p] * C[q] * h[p][q]
        for r in range(n):
            for s in range(n):
                energy += Q[p][r][q][s] * C[p] * C[q] * C[r] * C[s]
old_energy = energy

# Iterate to find self-consistent solution
for i in range(1000):
    # Create Fock matrix
    F = np.zeros((n, n))
    for p in range(n):
        for q in range(n):
            F[p][q] = h[p][q]
            for r in range(n):
                for s in range(n):
                    F[p][q] += Q[p][r][q][s] * C[r] * C[s]

    # Diagonalize overlap matrix to solve the generalized eigenvalue problem
    d, U = np.linalg.eigh(S)
    d[d < 1e-12] = 1e-12
    V = U @ np.diag(1 / np.sqrt(d))

    # Transform Fock matrix to new basis
    F_prime = V.T @ (F @ V)
    Eprime, Cprime = np.linalg.eigh(F_prime)

    # Transform back to original basis
    C = V @ Cprime[:, 0]

    # Normalize coefficients again
    norm2 = 0
    for p in range(n):
        for q in range(n):
            norm2 += C[p] * S[p][q] * C[q]
    C /= np.sqrt(norm2)

    # Compute new energy
    energy = 0
    for p in range(n):
        for q in range(n):
            energy += 2 * C[p] * C[q] * h[p][q]
            for r in range(n):
                for s in range(n):
                    energy += Q[p][r][q][s] * C[p] * C[q] * C[r] * C[s]

    # convergence criterion
    if (27.2114 * abs(energy - old_energy) < 1e-5):
        break

    old_energy = energy

print(f"Ground state energy: {energy:.7f} [Ha](ideally, it should be -2.8551716[Ha])")
print(f"C-parameters: {C}")

# Compute and save wavefunction
N = 1000
r_lin = np.linspace(0, 5, N)
phi = np.abs(np.zeros_like(r_lin))

for p in range(n):
    phi += C[p] * np.exp(-alpha[p] * r_lin**2)

# Save alpha parameters and C-parameters, with clear titles and alignment
with open('./A5/task1_helium_wavefunction.txt', 'w') as file:
    file.write("---- Calculated C-parameters for the helium ground state wavefunction ----\n")
    file.write("This file contains the C-parameters for the helium wavefunction based on four Gaussians as a basis, each on the form: C * exp(-alpha * r^2)\n\n")
    
    file.write("---- Alpha Parameters ----\n")
    for i in range(n):
        file.write(f"alpha[{i}] = {alpha[i]:.6f}\n") 

    file.write("\n---- C Parameters (Coefficients) ----\n")
    for i in range(n):
        file.write(f"C[{i}] = {C[i]:.6f}\n")
    
    file.write("\n----  RESULTS ----\n")
    file.write(f"Number of points in discretized radial coordinate: {N}\n")
    file.write(f"Ground state energy of Helium : {energy:.7f} [Ha]\n" )  


# task 2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Function to create the second derivative matrix using finite differences
def create_matrix_D2_finite_difference(N, h):
    """
    Creates the matrix for the second derivative using finite differences.
    Parameters:
        N (int): Number of grid points.
        h (float): Grid spacing.
    
    Returns:
        D2 (numpy array): Matrix for the second derivative.
    """
    D2 = np.zeros((N, N))
    i, j = np.indices(D2.shape)

    # Operator matrix for numerical second derivative
    # Formula: d2y/dx2 = (y(k-1) - 2y(k) + y(k+1)) / dx^2
    D2[i == j] = -2 / h**2
    D2[abs(i - j) == 1] = 1 / h**2

    return D2

# Solve Poisson's equation d2U/dr2 = -u2/r using finite differences
def solve_poisson(r, u):
    """
    Solves Poisson's equation in radial coordinates using finite differences.
    Parameters:
        r (numpy array): Radial grid points.
        u (numpy array): Source term for Poisson's equation.
    
    Returns:
        U (numpy array): Potential solution to Poisson's equation.
    """
    # Grid spacing
    h = r[1] - r[0]
    N = len(r)

    # Create second derivative matrix
    D2 = create_matrix_D2_finite_difference(N, h)

    # Solve Poisson's equation with boundary conditions U(0) = 0, U(r_max) = 1
    U_0 = np.linalg.solve(D2, -u**2 / r)

    # Fix boundary conditions
    U = U_0 + r / np.max(r)

    return U

if __name__ == "__main__":

    # Define the radial grid
    N = 1000
    start, end = 0, 10
    r = np.linspace(start, end, N+1)[1:]  # Exclude r = 0 to avoid singularity
    h = r[1] - r[0]  # Grid spacing

    # Calculate the electron density for hydrogen's ground state
    electron_density = np.exp(-2 * r) / (np.pi)  # 1/(pi * a0^3) in normalized form for hydrogen

    # Solve Poisson's equation to obtain the potential
    u = np.sqrt(4 * np.pi * electron_density) * r
    U = solve_poisson(r, u)
    V_sH = U / r  # Hartree potential

    # Calculate the theoretical Hartree potential
    V_Hartree = 1.0 / r - (1.0 + 1.0 / r) * np.exp(-2.0 * r)

    # Plot the results
    plt.figure(figsize=(10, 6))
    plt.plot(r, V_Hartree, color='blue', marker='*', linestyle='', label='Theoretical Hartree potential')
    plt.plot(r, V_sH, color='red', marker='', linestyle='--', label='Calculated Hartree potential')
    plt.title('Hartree potential for hydrogen atom')
    plt.xlabel('Radial distance r (atomic units)')
    plt.ylabel('Energy of potential (atomic units)')
    plt.grid(True)
    plt.legend()
    plt.savefig(f'./A5/task2_Hartree_potential_N={N}.png')
    plt.show()

    # Save data to CSV for further analysis or plotting
    np.savetxt(f'./A5/task2_Hartree_potential_N={N}.csv', 
               np.column_stack([r, V_Hartree, V_sH]), 
               header="Radial distance (atomic units), Theoretical Hartree potential, Calculated Hartree potential", 
               delimiter=",", comments="")


# task 3

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import trapezoid

# Function to create the second derivative matrix using finite differences
def create_matrix_D2_finite_difference(N, h):
    """
    Creates the matrix for the second derivative using finite differences.
    Parameters:
        N (int): Number of grid points.
        h (float): Grid spacing.
    Returns:
        D2 (numpy array): Matrix for the second derivative.
    """
    D2 = np.zeros((N, N))
    i, j = np.indices(D2.shape)

    # Operator matrix for numerical second derivative
    # Formula: d2y/dx2 = (y(k-1) - 2y(k) + y(k+1)) / dx^2
    D2[i == j] = -2 / h**2
    D2[abs(i - j) == 1] = 1 / h**2

    return D2

# Normalize the radial wavefunction such that its total probability is 1
def normalize_radial_wavefunction(psi, r):
    """
    Normalize the radial wavefunction such that the total probability equals 1.
    Parameters:
        psi (numpy array): The radial wavefunction.
        r (numpy array): The radial grid.
    Returns:
        normalized_psi (numpy array): The normalized radial wavefunction.
    """
    norm = trapezoid(psi**2, r)  # Integrate psi^2 over r
    return psi / np.sqrt(norm)

# Compute the total probability of the radial wavefunction
def total_probability_of_radial_wavefunction(psi, r):
    """
    Compute the total probability of the radial wavefunction.
    Parameters:
        psi (numpy array): The radial wavefunction.
        r (numpy array): The radial grid.
    Returns:
        probability (float): The total probability, which should be 1 for a normalized wavefunction.
    """
    return trapezoid(psi**2, r)

# Solve Poisson's equation d2U/dr2 = -u2/r using finite differences
def solve_poisson(r, u):
    """
    Solves Poisson's equation in radial coordinates using finite differences.
    Parameters:
        r (numpy array): Radial grid points.
        u (numpy array): Source term for Poisson's equation.
    
    Returns:
        U (numpy array): Potential solution to Poisson's equation.
    """
    h = r[1] - r[0]
    N = len(r)

    # Create second derivative matrix
    D2 = create_matrix_D2_finite_difference(N, h)

    # Solve Poisson's equation with boundary conditions U(0) = 0, U(r_max) = 1
    U_0 = np.linalg.solve(D2, -u**2 / r)

    # Fix boundary conditions
    U = U_0 + r / np.max(r)

    return U

def solve_kohn_sham(r, potential):
    """
    Solves the radial Kohn-Sham equation for a given potential.
    Parameters:
        r (numpy array): Radial grid points.
        potential (numpy array): The potential for the Kohn-Sham equation.
    
    Returns:
        eps (float): The energy of the lowest state.
        u (numpy array): The corresponding wavefunction for the lowest energy.
    """
    h = r[1] - r[0]
    N = len(r)
    
    D2 = create_matrix_D2_finite_difference(N, h)
    potential_matrix = np.diag(potential)

    # Solve Kohn-Sham matrix equation
    eps_vec, u_mat = np.linalg.eigh(-0.5 * D2 + potential_matrix)

    # eigh will sort eigenvalues in ascending order
    eps = eps_vec[0] 
    u = u_mat[:,0]

    if u[0] < 0: u = -u
    norm2 = trapezoid(u**2, r)
    u /= np.sqrt(norm2)

    return eps, u

# Define the radial grid (excluding r=0 to avoid singularity)
N = 1000 
linspace_start, linspace_end = 0, 10
r = np.linspace(linspace_start, linspace_end, N+1)[1:]
h = r[1] - r[0]

# Hydrogen atom functions: Direct implementation in the main loop
a0 = 1  # Bohr radius [a.u]

# Ground state wavefunction of hydrogen
def ground_state_wavefunction(r, a0=1):
    return (1 / np.sqrt(np.pi)) * (1 / a0**(3/2)) * np.exp(-r / a0)

# Ground state energy of hydrogen (theoretically -0.5 [a.u])
def ground_state_energy():
    return -0.5

# Calculate electron density for hydrogen's ground state
electron_density = np.exp(-2 * r) / (np.pi * a0**3)  # 1/(pi * a0^3) in normalized form for hydrogen

# Solve Poisson's equation to obtain the potential
u = np.sqrt(4 * np.pi * electron_density) * r
U = solve_poisson(r, u)
V_sH = U / r  # Hartree potential

# Solve for hydrogen atom as a test case (Coulomb potential)
potential = -1 / r  # Potential for hydrogen atom
E_hydrogen, u_hydrogen = solve_kohn_sham(r, potential)

# Compute wavefunction from u
psi_hydrogen = u_hydrogen / (np.sqrt(4 * np.pi) * r)  # Normalize wavefunction in spherical coordinates
psi_hydrogen = normalize_radial_wavefunction(psi_hydrogen, r)

# Compare with theoretical wavefunction for hydrogen atom
psi_hydrogen_theoretical = ground_state_wavefunction(r)

# Print to verify reasonability
print("\nHydrogen")
print(f"Ground state energy: {E_hydrogen:.7f} (theoretically: {ground_state_energy():.7f} [a.u] )")
print(f"Wavefunction's total probability (theoretical): {total_probability_of_radial_wavefunction(psi_hydrogen_theoretical, r):.6f} (normalized if 1)")
print(f"Wavefunction's total probability (calculated): {total_probability_of_radial_wavefunction(psi_hydrogen, r):.6f} (normalized if 1)")

# Plot the results
plt.figure(figsize=(10, 6))
plt.plot(r, psi_hydrogen_theoretical, color='blue', marker='^', linestyle='-', label='Theoretical hydrogen wavefunction')
plt.plot(r, psi_hydrogen, color='red', marker='.', linestyle='--', label='Calculated hydrogen wavefunction')
plt.xlabel('Radial distance r [a.u]')
plt.ylabel('Wavefunction')
plt.title('Hydrogen atom ground state wavefunction')
plt.grid()
plt.legend()
plt.savefig(f'./A5/task3_hydrogen_wavefunction_N={N}.png')

# Save data to CSV for further analysis or plotting
output_path_energy = f'./A5/task3_hydrogen_energy_N={N}.txt'  
with open(output_path_energy, 'w') as file:
    file.write(f"Calculated ground state energy of hydrogen: {E_hydrogen:.7f} [a.u]\n")
    file.write(f"Theoretical ground state energy of hydrogen: - 0.5 [a.u]\n")
    file.write(f"Number of points in discretized radial coordinate: {N}")


# task 4-7

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import trapezoid
from task2 import solve_poisson
from task3 import solve_kohn_sham
import csv

def save_to_csv(filename, r, psi):
    with open(filename, 'w', newline='') as csvfile:
        fieldnames = ['Radial Distance r (a.u.)', 'Computed Wavefunction']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

        writer.writeheader()

        for i in range(len(r)):
            writer.writerow({
                'Radial Distance r (a.u.)': r[i],
                'Computed Wavefunction': psi[i],
            })
        
def wavefunction_anzats(r):
    # constants from taks 1
    alpha  = np.array([0.297104, 1.236745, 5.749982, 38.216677])
    C      = np.array([-0.146876, -0.393152, -0.411198, -0.262007])

    psi = np.zeros_like(r)
    for p in range(0, len(C)):
        psi += C[p] * np.exp( - alpha[p] * r**2 )
    
    return psi

def n_s_from_u(r, u):
    return u**2 / (4 * np.pi * r**2)

def eps_exchange(n):
    return - (3/4) * (3 * n / np.pi)**(1/3)

def V_exchange(n):
    return - (3 * n / np.pi)**(1/3)


def eps_correlation(n):
    #constants from task description
    A = 0.0311
    B = -0.048
    C = 0.0020
    D = -0.0116
    gamma = -0.1423
    beta_1 = 1.0529
    beta_2 = 0.3334

    r_s = (3 / (4 * np.pi * n) )**(1/3)
    return np.where(r_s >=1, 
        gamma / (1 + beta_1*np.sqrt(r_s) + beta_2*r_s),
        A*np.log(r_s) + B + C*r_s*np.log(r_s) + D*r_s
    )

def V_correlation(n):
    #constants from task description
    A = 0.0311
    B = -0.048
    C = 0.0020
    D = -0.0116
    gamma = -0.1423
    beta_1 = 1.0529
    beta_2 = 0.3334

    r_s = (3 / (4 * np.pi * n) )**(1/3)
    return np.where(r_s >=1, 
        gamma * 
        (3 + 3.5 * beta_1 * np.sqrt(r_s) + 4 * beta_2 * r_s) /
        (3 * (1 + beta_1 * np.sqrt(r_s) + beta_2 * r_s))
        ,
        A * (np.log(r_s) - 1/3) + 
        B + 
        C * (2 * r_s * np.log(r_s) - r_s) / 3 + 
        2 * D * r_s / 3
    )

def find_scf_wavefunction(rmax, N, include_exchange, include_correlation):
    r = np.linspace(0, rmax, N+1)[1:]
    u = np.sqrt(4 * np.pi) * r * wavefunction_anzats(r)

    E = 0
    E_old = 0
    energy_list = []  # Store the energy at each iteration
    for i in range(100):
        U = solve_poisson(r, u)
        V_sH = U / r
        
        if include_exchange:
            V_H = 2 * V_sH
        else:
            V_H = V_sH

        n = 2 * n_s_from_u(r, u)

        eps_xc = np.zeros_like(r)
        V_xc = np.zeros_like(r)
        if include_exchange:
            eps_xc += eps_exchange(n)
            V_xc += V_exchange(n)
        if include_correlation:
            eps_xc += eps_correlation(n)
            V_xc += V_correlation(n)

        potential = - 2.0 / r + V_H + V_xc
        eps, u = solve_kohn_sham(r, potential)

        E_old = E
        E = 2 * eps - 2 * trapezoid(u**2 * (0.5 * V_H + V_xc - eps_xc), r)
        print(f"Iteration {i}: Energy = {E:.6f} [a.u.]")
        energy_list.append(E)

        if abs(E - E_old) < 1e-5:
            break
        
    psi = u / (np.sqrt(4 * np.pi) * r)
    print(f"Converged in {i} iterations")
    return E, r, psi, energy_list

# task4 DFT self-consistent field without exchange and correlation
# only the Hartree potential is included
# task = 4
# include_exchange = False
# include_correlation = False

# task5 DFT self-consistent field with exchange but without correlation
# task = 5
# include_exchange = True
# include_correlation = False

# task6 DFT self-consistent field with correlation but without exchange
# task = 6
# include_exchange = False
# include_correlation = True

# task7 DFT self-consistent field with exchange and correlation
task = 7
include_exchange = True
include_correlation = True

# E, r, psi = find_scf_wavefunction(30, 6000, include_exchange, include_correlation)
# print(f"Ground state energy: {E:.6f} (a.u.)")

# plt.plot(r, psi, color='black', marker='', linestyle='-', label='Computed helium wavefunction')
# plt.xlabel('Radial distance r (atomic units)')
# plt.ylabel('Wavefunction')
# plt.grid()
# plt.legend()
# plt.show()
a0 = 1
E, r, psi, energy_list = find_scf_wavefunction(30, 6000, include_exchange, include_correlation)
save_to_csv(f'./A5/task{task}_helium_wavefunction_and_energy.csv', r, psi)

# Theoretical Hydrogen atom wavefunction  for comparison
psi_theoretical = (1 / np.sqrt(np.pi)) * (1 / a0**(3/2)) * np.exp(-r / a0)
print(f"Task{task}: Ground state energy: {E:.6f} [a.u.]")

# Plot energy convergence over iterations
plt.figure(figsize=(10, 6))
plt.plot(range(len(energy_list)), energy_list, color='blue', marker='o', linestyle='-', label="Energy convergence")
plt.xlabel('Iteration')
plt.ylabel('Energy [a.u.]')
plt.title(f'Task{task}:Energy Convergence in SCF Iterations with exchange= {include_exchange}, correlation = {include_correlation}')
plt.grid(True)
plt.legend()
plt.savefig(f"./A5/task{task}_energy_convergence_plot.png")  

# Plot the computed wavefunction
plt.figure(figsize=(10, 6))
plt.plot(r, psi_theoretical, color='blue', marker='^', linestyle='', label='Theoretical helium wavefunction')
plt.plot(r, psi, color='red', marker='', linestyle='-', label='Computed helium wavefunction')
plt.xlabel('Radial distance r [a.u.]')
plt.ylabel('Wavefunction')
plt.title(f'Task{task}: Helium atom ground state wavefunction with exchange= {include_exchange}, correlation = {include_correlation}')
plt.grid()
plt.legend()
plt.savefig(f'./A5/task{task}_helium_wavefunction_exchange={include_exchange}_correlation={include_correlation}.png')


with open(f'./A5/task{task}_helium_energy.txt', 'w') as file:
    file.write(f"Calculated ground state energy of helium: {E:.8f} [a.u.]\n")
    file.write(f"Number of points in discretized radial coordinate: 6000 \n")
    file.write(f"Exchange included: {include_exchange}\n")
    file.write(f"Correlation included: {include_correlation}\n")
    file.write(f"Wavefunction's total probability: {trapezoid(psi**2 * 4 * np.pi * r**2, r):.6f} (normalized if 1)\n")


# task 8

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def read_wavefunction(filename):
    data = np.loadtxt(filename, delimiter=',', skiprows=1)  
    r = data[:, 0]  
    psi = data[:, 1] 
    return r, psi

r_task4, psi_task4 = read_wavefunction("./A5/task4_helium_wavefunction_and_energy.csv")
r_task5, psi_task5 = read_wavefunction("./A5/task5_helium_wavefunction_and_energy.csv")
r_task6, psi_task6 = read_wavefunction("./A5/task6_helium_wavefunction_and_energy.csv")
r_task7, psi_task7 = read_wavefunction("./A5/task7_helium_wavefunction_and_energy.csv")


plt.figure(figsize=(10, 6))
plt.plot(r_task4, psi_task4, label='Task 4: No exchange, no correlation', color='blue')
plt.plot(r_task5, psi_task5, label='Task 5: Exchange, no correlation', color='red')
plt.plot(r_task6, psi_task6, label='Task 6: No exchange, correlation', color='green')
plt.plot(r_task7, psi_task7, label='Task 7: Exchange and correlation', color='purple')

plt.xlabel('Radial distance r (a.u.)')
plt.ylabel('Wavefunction')
plt.title('Helium Atom Ground State Wavefunction for Different Tasks')
plt.legend()
plt.grid(True)
plt.savefig(f'./A5/task8_helium_wavefunction_comparison.png')
